In [1]:
# Colab-Setup — wird in lokalen Umgebungen automatisch übersprungen.
import sys

if 'google.colab' in sys.modules:
    import os

    REPO_DIR = '/content/Wassermengenwirtschaft_und_Klimawandel'

    # 1) Repository klonen (idempotent)
    if not os.path.isdir(REPO_DIR):
        print('Colab-Setup: Repository wird geklont ...')
        get_ipython().system('git clone https://github.com/gjohnen1/Wassermengenwirtschaft_und_Klimawandel.git ' + REPO_DIR)
    else:
        print('Colab-Setup: Repository ist bereits vorhanden.')

    get_ipython().run_line_magic('cd', REPO_DIR)

    # 2) Abhängigkeiten installieren
    print('Colab-Setup: Installiere requirements.txt (rtc-tools, casadi, plotly, pysheds, ...)')
    get_ipython().run_line_magic('pip', 'install -q -r requirements.txt')

    # 3) ipywidgets-Rendering in Colab explizit aktivieren — sonst wird die
    #    Goal-Editor-GUI weiter unten nur als statische Textzeile angezeigt.
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()

    print()
    print('Colab-Setup fertig.')
    print("Falls Colab jetzt 'You must restart the runtime' meldet:")
    print("  1. Runtime > Restart runtime")
    print("  2. danach diese Zelle erneut ausführen (die Pakete bleiben installiert)")
else:
    print('Nicht in Colab erkannt — lokaler/Binder-Modus wird genutzt.')


Nicht in Colab erkannt — lokaler/Binder-Modus wird genutzt.


In [2]:
# Smoke-Check: läuft RTC-Tools im Kernel?
import sys, platform, importlib

print(f"Plattform:  {platform.system()} ({platform.machine()})")
print(f"Python:     {sys.version.split()[0]}")

missing = []
for pkg in ("rtctools", "casadi", "pymoca", "rtctools_interface",
            "pandas", "plotly", "ipywidgets"):
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    raise ImportError(
        "Fehlende Pakete: " + ", ".join(missing) +
        ". Bitte 'pip install -r requirements.txt' in der aktiven Umgebung ausführen."
    )

import rtctools, casadi
print(f"rtctools:   {rtctools.__version__}")
print(f"casadi:     {casadi.__version__}")
print("Alle Kernel-Pakete vorhanden — bereit für die Optimierung.")


Plattform:  Windows (AMD64)
Python:     3.10.16


rtctools:   2.7.3
casadi:     3.7.2
Alle Kernel-Pakete vorhanden — bereit für die Optimierung.


In [3]:
import os
import sys
import shutil
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    """Repo-Wurzel finden: enthält 'Inhalt/simple_reservoir/BlueRiver2'."""
    for p in [start, *start.parents]:
        if (p / "Inhalt" / "simple_reservoir" / "BlueRiver2").exists():
            return p
    raise FileNotFoundError(
        "Repository-Wurzel mit Inhalt/simple_reservoir/BlueRiver2 nicht gefunden."
    )

REPO_ROOT     = find_repo_root(Path.cwd())
BLUERIVER_DIR = REPO_ROOT / "Inhalt" / "simple_reservoir" / "BlueRiver2"
DATA_ROOT     = REPO_ROOT / "Inhalt" / "Notebook_Daten" / "RTC_BlueRiver"

SCENARIO_SHORT = DATA_ROOT / "ShortTerm" / "input"
SCENARIO_LONG  = DATA_ROOT / "LongTerm"  / "input"
DATA_MODEL     = DATA_ROOT / "model"

CASE_DIR    = DATA_ROOT / "runtime_case"
CASE_INPUT  = CASE_DIR / "input"
CASE_MODEL  = CASE_DIR / "model"
CASE_SRC    = CASE_DIR / "src"
CASE_XSD    = CASE_DIR / "xsd"
CASE_OUTPUT = CASE_DIR / "output"

# Statt eines separaten Sub-venv läuft das Modell direkt im Kernel-Interpreter.
PY = sys.executable

print(f"Repo root:     {REPO_ROOT}")
print(f"BlueRiver:     {BLUERIVER_DIR}")
print(f"Data root:     {DATA_ROOT}")
print(f"Runtime case:  {CASE_DIR}")
print(f"Python:        {PY}")


Repo root:     C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel
BlueRiver:     C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2
Data root:     C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver
Runtime case:  C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Python:        C:\Users\grego\anaconda3\envs\hausarbeit_wb4\python.exe


In [4]:
required_blue = [
    BLUERIVER_DIR / "src"   / "BlueRiver.py",
    BLUERIVER_DIR / "model" / "BlueRiver.mo",
    BLUERIVER_DIR / "xsd"   / "rtcDataConfig.xsd",
    BLUERIVER_DIR / "xsd"   / "rtcSharedTypes.xsd",
]

required_data = [
    DATA_MODEL / "reservoirs.csv",
    DATA_MODEL / "volumelevel.csv",
]

required_short = [
    SCENARIO_SHORT / "goal_table.csv",
    SCENARIO_SHORT / "plot_table.csv",
    SCENARIO_SHORT / "rtcDataConfig.xml",
    SCENARIO_SHORT / "rtcParameterConfig.xml",
    SCENARIO_SHORT / "timeseries_import.csv",
    SCENARIO_SHORT / "timeseries_import.xml",
]

missing = [p for p in (required_blue + required_data + required_short) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Folgende Dateien fehlen:\n" + "\n".join(str(p) for p in missing)
    )

print("Alle Pflichtdateien vorhanden.")
print(f"BlueRiver.py:  {required_blue[0]}")
print(f"BlueRiver.mo:  {required_blue[1]}")


Alle Pflichtdateien vorhanden.
BlueRiver.py:  C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2\src\BlueRiver.py
BlueRiver.mo:  C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\simple_reservoir\BlueRiver2\model\BlueRiver.mo


In [5]:
import shutil
import stat
import time

def _on_rm_error(func, path, exc_info):
    # Windows/OneDrive: read-only Attribute entfernen und erneut versuchen
    try:
        os.chmod(path, stat.S_IWRITE)
        func(path)
    except Exception:
        pass

def _remove_path(path: Path):
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path, onerror=_on_rm_error)
    else:
        try:
            os.chmod(path, stat.S_IWRITE)
        except Exception:
            pass
        path.unlink(missing_ok=True)

def _clear_dir(path: Path, retries: int = 5, delay_s: float = 0.4):
    path.mkdir(parents=True, exist_ok=True)
    for child in list(path.iterdir()):
        for attempt in range(1, retries + 1):
            try:
                _remove_path(child)
                break
            except PermissionError:
                if attempt == retries:
                    raise
                time.sleep(delay_s * attempt)

def prepare_case(scenario_input: Path):
    # Frische Ordnerstruktur (Output bleibt erhalten, damit figures/<RUN_TAG>/
    # zwischen Konfigurationen nicht überschrieben werden).
    for p in [CASE_INPUT, CASE_MODEL, CASE_SRC, CASE_XSD]:
        _clear_dir(p)
    CASE_OUTPUT.mkdir(parents=True, exist_ok=True)

    shutil.copytree(BLUERIVER_DIR / "src",   CASE_SRC,   dirs_exist_ok=True)
    shutil.copytree(BLUERIVER_DIR / "model", CASE_MODEL, dirs_exist_ok=True)
    shutil.copytree(BLUERIVER_DIR / "xsd",   CASE_XSD,   dirs_exist_ok=True)

    shutil.copy2(DATA_MODEL / "reservoirs.csv",  CASE_MODEL / "reservoirs.csv")
    shutil.copy2(DATA_MODEL / "volumelevel.csv", CASE_MODEL / "volumelevel.csv")

    shutil.copytree(scenario_input, CASE_INPUT, dirs_exist_ok=True)

    print("Case vorbereitet:", CASE_DIR)
    print("Input:",  CASE_INPUT)
    print("Output:", CASE_OUTPUT)


In [6]:
prepare_case(SCENARIO_SHORT)
print("ShortTerm-Szenario ist aktiv.")


Case vorbereitet: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Input: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\input
Output: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output
ShortTerm-Szenario ist aktiv.


In [7]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

GOAL_TABLE_RUNTIME = CASE_INPUT / "goal_table.csv"

if not GOAL_TABLE_RUNTIME.exists():
    raise FileNotFoundError(f"Fehlende Datei: {GOAL_TABLE_RUNTIME}")

_editor_df = pd.read_csv(GOAL_TABLE_RUNTIME)
_row_controls = {}


def _bool_from_cell(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        return v.strip().lower() in {"1", "true", "yes", "y"}
    return bool(int(v)) if isinstance(v, (int, float)) else bool(v)


def _int_or_default(v, default=0):
    try:
        return int(float(v))
    except Exception:
        return default


def _float_or_default(v, default=1.0):
    try:
        return float(v)
    except Exception:
        return default


def _build_row(idx, row):
    id_val = _int_or_default(row.get("id"), idx)
    state = str(row.get("state", ""))
    goal_type = str(row.get("goal_type", ""))

    meta = widgets.HTML(value=f"<b>ID {id_val}</b> | {state} | {goal_type}",
                        layout=widgets.Layout(width="360px"))
    active = widgets.Checkbox(value=_bool_from_cell(row.get("active", 0)),
                              description="active", indent=False,
                              layout=widgets.Layout(width="90px"))
    priority = widgets.BoundedIntText(value=_int_or_default(row.get("priority"), 1),
                                      min=0, max=100, description="prio",
                                      layout=widgets.Layout(width="140px"))
    weight = widgets.FloatText(value=_float_or_default(row.get("weight"), 1.0),
                               description="weight",
                               layout=widgets.Layout(width="160px"))
    order = widgets.BoundedIntText(value=max(1, _int_or_default(row.get("order"), 1)),
                                   min=1, max=10, description="order",
                                   layout=widgets.Layout(width="150px"))

    box = widgets.HBox([meta, active, priority, weight, order])
    _row_controls[idx] = {"active": active, "priority": priority,
                          "weight": weight, "order": order}
    return box


def _collect_df_from_controls(base_df):
    df = base_df.copy()
    for idx in df.index:
        c = _row_controls[idx]
        df.at[idx, "active"]   = 1 if c["active"].value else 0
        df.at[idx, "priority"] = int(c["priority"].value)
        df.at[idx, "weight"]   = float(c["weight"].value)
        df.at[idx, "order"]    = int(c["order"].value)
    return df


status = widgets.Output(layout=widgets.Layout(border="1px solid #ddd",
                                              padding="6px",
                                              max_height="160px",
                                              overflow="auto"))

save_btn = widgets.Button(description="Goals speichern", button_style="success",
                          icon="save",
                          layout=widgets.Layout(width="220px"))


def _on_save(_):
    base_df = pd.read_csv(GOAL_TABLE_RUNTIME)
    new_df = _collect_df_from_controls(base_df)
    new_df.to_csv(GOAL_TABLE_RUNTIME, index=False)
    with status:
        clear_output(wait=True)
        print(f"Gespeichert nach: {GOAL_TABLE_RUNTIME}")
        print()
        print(new_df[["id", "state", "active", "priority", "weight", "order"]].to_string(index=False))


save_btn.on_click(_on_save)

rows = [_build_row(idx, row) for idx, row in _editor_df.iterrows()]

display(widgets.VBox([
    widgets.HTML(
        "<b>Goal-Editor</b> — Werte anpassen und auf <i>Goals speichern</i> klicken. "
        f"Die Datei wird unter <code>{GOAL_TABLE_RUNTIME}</code> abgelegt."
    ),
    widgets.VBox(rows),
    save_btn,
    status,
]))

with status:
    clear_output(wait=True)
    print(f"Editor bereit. Aktuelle Datei: {GOAL_TABLE_RUNTIME}")


In [8]:
import subprocess

# Tag für diesen Optimierungslauf — wird später als Ordnername für die exportierten
# Bilder verwendet. Ändern Sie den Tag, bevor Sie die Goal-Tabelle modifizieren
# und neu rechnen, um die Ergebnisse vergleichbar abzulegen.
RUN_TAG = "shortterm_default"

run_cmd = [PY, "BlueRiver.py"]
print(f"RUN_TAG: {RUN_TAG}")
print("Starte:", " ".join(run_cmd))

run_env = os.environ.copy()
run_env["MPLBACKEND"] = "Agg"
result = subprocess.run(run_cmd, cwd=CASE_SRC, env=run_env,
                        capture_output=True, text=True)
combined_log = (result.stdout or "") + "\n" + (result.stderr or "")
log_path = CASE_DIR / "venv-log.txt"
log_path.write_text(combined_log, encoding="utf-8")

print("Return code:", result.returncode)
print("Log gespeichert in:", log_path)
print("\n--- Letzte Logzeilen ---")
print("\n".join(combined_log.splitlines()[-30:]))

if result.returncode != 0:
    raise RuntimeError(
        "BlueRiver-Lauf fehlgeschlagen. Siehe venv-log.txt und Troubleshooting-Abschnitt."
    )


RUN_TAG: shortterm_default
Starte: C:\Users\grego\anaconda3\envs\hausarbeit_wb4\python.exe BlueRiver.py


Return code: 0
Log gespeichert in: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\venv-log.txt

--- Letzte Logzeilen ---
2026-04-21 08:06:00,843 INFO Mapping
2026-04-21 08:06:00,844 INFO Composing NLP segment
2026-04-21 08:06:00,845 INFO Creating NLP dictionary
2026-04-21 08:06:00,845 INFO Done transcribing problem
CasADi - 2026-04-21 08:06:00 WARNING("CasADi was not compiled with WITH_OPENMP=ON. Falling back to serial evaluation.") [.../casadi/core/map.cpp:406]
CasADi - 2026-04-21 08:06:00 WARNING("CasADi was not compiled with WITH_OPENMP=ON. Falling back to serial evaluation.") [.../casadi/core/map.cpp:406]
2026-04-21 08:06:00,854 INFO Calling solver
2026-04-21 08:06:00,947 INFO Solver succeeded with status Solve_Succeeded (0.09028 seconds).
2026-04-21 08:06:00,947 INFO Done with optimize()
2026-04-21 08:06:00,950 INFO Extracting results
2026-04-21 08:06:00,951 INFO Done extrac

In [9]:
import xml.etree.ElementTree as ET
import pandas as pd

out_csv = CASE_OUTPUT / "timeseries_export.csv"
out_diag = CASE_OUTPUT / "diag.xml"
perf_dir = CASE_OUTPUT / "performance_metrics"
run_log = CASE_DIR / "venv-log.txt"

if not out_csv.exists():
    raise FileNotFoundError(f"Fehlt: {out_csv}")

df = pd.read_csv(out_csv, parse_dates=["time"])
required_cols = ["time", "TroutLake_V", "TroutLake_Q_out", "RiverCity_Q"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError("Fehlende Spalten in timeseries_export.csv: " + ", ".join(missing_cols))
if df.empty:
    raise ValueError("timeseries_export.csv ist leer.")

diag_info = "Keine Diag-Information gefunden."
if out_diag.exists():
    root = ET.parse(out_diag).getroot()
    lines = list(root)
    levels = {"1": 0, "2": 0, "3": 0}
    for line in lines:
        lvl = line.attrib.get("level")
        if lvl in levels:
            levels[lvl] += 1
    diag_info = f"diag.xml vorhanden. Level-Count: {levels}"
elif perf_dir.exists() and list(perf_dir.glob("*.csv")):
    diag_info = f"diag.xml fehlt, aber performance_metrics vorhanden ({len(list(perf_dir.glob('*.csv')))} Dateien)."
elif run_log.exists():
    log_text = run_log.read_text(encoding="utf-8", errors="ignore")
    if "Traceback" in log_text:
        raise RuntimeError("venv-log.txt enthaelt einen Traceback. Lauf pruefen.")
    if "Done goal programming" in log_text:
        diag_info = "diag.xml fehlt, aber Lauf laut venv-log.txt erfolgreich (Done goal programming)."
    else:
        raise FileNotFoundError("Weder diag.xml noch performance_metrics gefunden. Laufpruefung unvollstaendig.")
else:
    raise FileNotFoundError("Weder diag.xml noch performance_metrics/venv-log.txt gefunden.")

print("CSV ok. Zeilen:", len(df))
print(diag_info)
short_term_df = df.copy()


CSV ok. Zeilen: 57
diag.xml fehlt, aber performance_metrics vorhanden (4 Dateien).


In [10]:
# Plotly-Renderer auf CDN-basiert setzen — sonst landet plotly.js (~3 MB)
# pro Figur INLINE in der gerenderten HTML-Seite (Jupyter-Book-Build).
import plotly.io as pio
pio.renderers.default = "notebook_connected"

import json
from collections import OrderedDict

import pandas as pd
import plotly.graph_objects as go


CACHE_DIR = CASE_OUTPUT / "cached_results"
cache_files = sorted(CACHE_DIR.glob("*.json"), key=lambda p: p.stat().st_mtime)
if not cache_files:
    raise FileNotFoundError(f"Keine cached_results-Datei gefunden in: {CACHE_DIR}")

cache_path = cache_files[-1]
cache_data = json.loads(cache_path.read_text(encoding="utf-8"))

intermediate_results = cache_data.get("intermediate_results", [])
if not intermediate_results:
    raise ValueError("cached_results enthält keine intermediate_results.")


def _unwrap_array(value):
    if isinstance(value, dict) and "data" in value:
        return value["data"]
    return value


io_datetimes = cache_data["prio_independent_data"]["io_datetimes"]
time_index = pd.to_datetime([item["value"] for item in io_datetimes])

base_goals = {
    int(goal["goal_id"]): goal
    for goal in cache_data["prio_independent_data"]["base_goals"]
}
plot_rows = [item["value"] for item in cache_data["plot_options"]["plot_config"]]

color_map = {
    "TroutLake_V":         "royalblue",
    "RiverCity_Q":         "green",
    "TroutLake_Q_out":     "pink",
    "Alder_Inflow":        "olive",
    "TroutLake_Q_spill":   "brown",
    "TroutLake_Q_turbine": "violet",
}


def _add_target(fig, name, target_series):
    if target_series is None:
        return
    values = _unwrap_array(target_series)
    if not values:
        return
    numeric = [float(v) for v in values]
    if all(abs(v - numeric[0]) < 1e-12 for v in numeric):
        fig.add_hline(
            y=numeric[0], line_dash="dash", line_color="red",
            annotation_text=name,
            annotation_position="top right" if "max" in name.lower() else "bottom right",
        )
    else:
        fig.add_trace(go.Scatter(
            x=time_index[: len(numeric)], y=numeric, mode="lines",
            name=name, line=dict(color="red", dash="dash"),
        ))


def _build_figures_for_priority(priority, snapshot):
    """Return an OrderedDict[stem -> Figure] for a single priority snapshot."""
    timeseries_data = {
        key: _unwrap_array(value)
        for key, value in snapshot.get("timeseries_data", {}).items()
    }
    figs = OrderedDict()
    for row in plot_rows:
        goal_id = int(row["id"])
        goal = base_goals.get(goal_id)
        if goal is None:
            continue

        main_var = goal.get("state")
        extra_vars = row.get("variables_style_2", [])
        variables = [main_var] + [v for v in extra_vars if v != main_var]

        fig = go.Figure()
        for var in variables:
            values = timeseries_data.get(var)
            if values is None:
                continue
            y = [float(v) for v in values]
            x = time_index[: len(y)]
            fig.add_trace(go.Scatter(
                x=x, y=y, mode="lines", name=var,
                line=dict(color=color_map.get(var, None)),
            ))

        _add_target(fig, "Target max", goal.get("target_max_series"))
        _add_target(fig, "Target min", goal.get("target_min_series"))

        subtitle = row.get("custom_title", f"Goal {goal_id}")
        title = f"Priority {priority} | {subtitle}"
        fig.update_layout(
            title=title, xaxis_title="Time",
            yaxis_title=str(row.get("y_axis_title", "Value")).replace("$", ""),
            template="plotly_white",
        )

        # Stable filename stem
        safe_state = str(main_var or "var").replace("/", "_").replace(" ", "_")
        figs[f"priority_{priority:02d}__goal_{goal_id:03d}__{safe_state}"] = fig
    return figs


available_priorities = sorted(int(item.get("priority")) for item in intermediate_results)
figures = OrderedDict()
for prio in available_priorities:
    snapshot = next(item for item in intermediate_results if int(item.get("priority")) == prio)
    figures.update(_build_figures_for_priority(prio, snapshot))

print(f"Cached results: {cache_path.name}")
print(f"Verfügbare Prioritäten: {available_priorities}")
print(f"Anzahl erzeugter Figuren: {len(figures)}")

# Inline anzeigen
for fig in figures.values():
    fig.show()


Cached results: 1776751561.json
Verfügbare Prioritäten: [1, 3, 4, 9]
Anzahl erzeugter Figuren: 16


In [11]:
import threading
import shutil

EXPORT_HTML = True
EXPORT_PNG  = True
PNG_TIMEOUT_S       = 30   # Sekunden pro Figur
PNG_PROBE_TIMEOUT_S = 10   # Sekunden für einmalige Probe-Konvertierung

figures_dir = CASE_OUTPUT / "figures" / RUN_TAG
figures_dir.mkdir(parents=True, exist_ok=True)


def _write_png_with_timeout(fig, path, timeout_s):
    """Run fig.write_image in a daemon thread with a hard timeout."""
    error = []

    def _worker():
        try:
            fig.write_image(str(path), width=900, height=420, scale=2)
        except Exception as exc:
            error.append(exc)

    thread = threading.Thread(target=_worker, daemon=True)
    thread.start()
    thread.join(timeout=timeout_s)
    if thread.is_alive():
        return False, "TIMEOUT (kaleido/Chromium-Subprozess hängt)"
    if error:
        return False, str(error[0])
    return True, None


# Kaleido verfügbar + funktionsfähig auf diesem System?
can_export_png = False
if EXPORT_PNG:
    try:
        import kaleido  # noqa: F401
        if figures:
            probe_stem = next(iter(figures))
            probe_fig = figures[probe_stem]
            probe_path = figures_dir / "_kaleido_probe.png"
            ok, err = _write_png_with_timeout(probe_fig, probe_path, PNG_PROBE_TIMEOUT_S)
            if ok:
                can_export_png = True
            else:
                print(f"Hinweis: kaleido-Probe fehlgeschlagen ({err}).")
                print("        PNG-Export wird übersprungen — siehe Troubleshooting #6.")
                print("        HTML-Export läuft trotzdem.")
    except ImportError:
        print("Hinweis: 'kaleido' nicht installiert — überspringe PNG-Export.")
        print("        Installation: pip install 'kaleido==0.2.1'")


written = []
skipped_png = []

for stem, fig in figures.items():
    if EXPORT_HTML:
        html_path = figures_dir / f"{stem}.html"
        fig.write_html(str(html_path), include_plotlyjs="cdn")
        written.append(html_path.name)

    if can_export_png:
        png_path = figures_dir / f"{stem}.png"
        ok, err = _write_png_with_timeout(fig, png_path, PNG_TIMEOUT_S)
        if ok:
            written.append(png_path.name)
        else:
            skipped_png.append((stem, err))

# Bonus: rtc-tools produziert eigene matplotlib-PNGs (after_priority_*.png,
# final_results.png) direkt im figures/-Ordner. Wir kopieren sie zusätzlich in
# figures/<RUN_TAG>/, damit pro Lauf-Tag ein kompletter, in sich geschlossener
# Ergebnisordner entsteht.
native_png_root = CASE_OUTPUT / "figures"
native_pngs = [p for p in native_png_root.glob("*.png") if p.is_file()]
copied_native = 0
for png in native_pngs:
    target = figures_dir / png.name
    try:
        shutil.copy2(str(png), str(target))
        copied_native += 1
    except Exception as exc:
        print(f"  Konnte {png.name} nicht in {figures_dir.name}/ kopieren: {exc}")

print(f"Exportordner: {figures_dir}")
print(f"Geschrieben:  {len(written)} Plotly-Dateien")
for name in written[:12]:
    print(f"  - {name}")
if len(written) > 12:
    print(f"  ... und {len(written) - 12} weitere")

if copied_native:
    print(f"Zusätzlich {copied_native} matplotlib-PNG(s) aus rtc-tools in den RUN_TAG-Ordner kopiert.")

if skipped_png:
    print()
    print(f"PNG übersprungen: {len(skipped_png)} Figur(en) (HTML wurde trotzdem geschrieben)")


Hinweis: kaleido-Probe fehlgeschlagen (TIMEOUT (kaleido/Chromium-Subprozess hängt)).
        PNG-Export wird übersprungen — siehe Troubleshooting #6.
        HTML-Export läuft trotzdem.
Exportordner: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output\figures\shortterm_default
Geschrieben:  16 Plotly-Dateien
  - priority_01__goal_010__TroutLake_V.html
  - priority_01__goal_020__TroutLake_V.html
  - priority_01__goal_030__RiverCity_Q.html
  - priority_01__goal_060__TroutLake_Q_spill.html
  - priority_03__goal_010__TroutLake_V.html
  - priority_03__goal_020__TroutLake_V.html
  - priority_03__goal_030__RiverCity_Q.html
  - priority_03__goal_060__TroutLake_Q_spill.html
  - priority_04__goal_010__TroutLake_V.html
  - priority_04__goal_020__TroutLake_V.html
  - priority_04__goal_030__RiverCity_Q.html
  - priority_04__goal_060__TroutLake_Q_spill.html
  ... und 4 weitere
Zusätzlich 5 m

In [12]:
import plotly.graph_objects as go

RUN_LONGTERM = True

if not RUN_LONGTERM:
    print("LongTerm uebersprungen. Setzen Sie RUN_LONGTERM=True fuer die Erweiterung.")
else:
    prepare_case(SCENARIO_LONG)

    run_env_long = os.environ.copy()
    run_env_long["MPLBACKEND"] = "Agg"
    result_long = subprocess.run([PY, "BlueRiver.py"], cwd=CASE_SRC,
                                 env=run_env_long, capture_output=True, text=True)
    combined_long = (result_long.stdout or "") + "\n" + (result_long.stderr or "")
    (CASE_DIR / "venv-log.txt").write_text(combined_long, encoding="utf-8")
    if result_long.returncode != 0:
        raise RuntimeError("LongTerm-Lauf fehlgeschlagen. Siehe venv-log.txt")

    long_df = pd.read_csv(CASE_OUTPUT / "timeseries_export.csv", parse_dates=["time"])
    print("LongTerm-Zeilen:", len(long_df))

    fig_long_v = go.Figure()
    fig_long_v.add_trace(go.Scatter(x=short_term_df["time"], y=short_term_df["TroutLake_V"],
                                    mode="lines", name="ShortTerm",
                                    line=dict(color="royalblue")))
    fig_long_v.add_trace(go.Scatter(x=long_df["time"], y=long_df["TroutLake_V"],
                                    mode="lines", name="LongTerm",
                                    line=dict(color="orange")))
    fig_long_v.update_layout(title="Vergleich ShortTerm vs LongTerm: TroutLake_V",
                             xaxis_title="Time", yaxis_title="TroutLake_V [m^3]",
                             template="plotly_white")
    fig_long_v.show()

    fig_long_q = go.Figure()
    fig_long_q.add_trace(go.Scatter(x=short_term_df["time"], y=short_term_df["TroutLake_Q_out"],
                                    mode="lines", name="ShortTerm",
                                    line=dict(color="green")))
    fig_long_q.add_trace(go.Scatter(x=long_df["time"], y=long_df["TroutLake_Q_out"],
                                    mode="lines", name="LongTerm",
                                    line=dict(color="red")))
    fig_long_q.update_layout(title="Vergleich ShortTerm vs LongTerm: TroutLake_Q_out",
                             xaxis_title="Time", yaxis_title="TroutLake_Q_out [m^3/s]",
                             template="plotly_white")
    fig_long_q.show()

    # Auch die LongTerm-Vergleichsfiguren in den figures/<RUN_TAG>/-Ordner exportieren.
    long_dir = CASE_OUTPUT / "figures" / f"{RUN_TAG}_longterm_compare"
    long_dir.mkdir(parents=True, exist_ok=True)
    fig_long_v.write_html(str(long_dir / "compare__TroutLake_V.html"),
                          include_plotlyjs="cdn")
    fig_long_q.write_html(str(long_dir / "compare__TroutLake_Q_out.html"),
                          include_plotlyjs="cdn")
    print(f"LongTerm-Vergleichsfiguren als HTML exportiert nach: {long_dir}")


Case vorbereitet: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case
Input: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\input
Output: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output


LongTerm-Zeilen: 61


LongTerm-Vergleichsfiguren als HTML exportiert nach: C:\Users\grego\OneDrive - Universitaet Duisburg-Essen\GitHub\repos\Wassermengenwirtschaft_und_Klimawandel\Inhalt\Notebook_Daten\RTC_BlueRiver\runtime_case\output\figures\shortterm_default_longterm_compare
